> **Portfolio version.** Cell outputs and workspace-specific connection details have been removed. Configure the environment variables documented in the repository README before running on Databricks.


# Bronze ? Raw CSV Ingestion

Ingests seven CSV sources from the configured ADLS Gen2 raw container and writes auditable Delta Bronze tables.

- keep all source data values as strings
- file lineage with metadata columns
- minimal structural fixes needed for Delta storage
- source files where the true CSV header is not at row 1 -> fix
- keep multi-row headers raw when Silver needs the extra header context


In [ ]:
from datetime import datetime
import re

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

## 1. Configuration


In [ ]:
# Storage configuration (set these as Databricks cluster environment variables)
import os

STORAGE_ACCOUNT = os.environ.get("AZURE_STORAGE_ACCOUNT")
RAW_CONTAINER = os.environ.get("AZURE_RAW_CONTAINER", "raw")
RAW_PREFIX = os.environ.get("AZURE_RAW_PREFIX", "")

if not STORAGE_ACCOUNT:
    raise EnvironmentError("Set AZURE_STORAGE_ACCOUNT before running this notebook.")

adls_path = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{RAW_PREFIX}".rstrip("/")
raw_root = adls_path

bronze_schema = "cbda_bronze"
write_mode = "overwrite"
target_namespace = f"`{bronze_schema}`"

def anon(text):
    return str(text).replace(STORAGE_ACCOUNT, "***")

print(f"ADLS path: {anon(adls_path)}")
print(f"Target namespace: {target_namespace}")
print(f"Write mode: {write_mode}")

files = dbutils.fs.ls(raw_root)
print(f"Found {len(files)} items")
for f in files:
    print(f"  - {f.name}")


## 2. Dataset 


In [ ]:
DATASETS = [
    {
        "dataset_name": "mps_crime",
        "table_name": "bronze_mps_crime",
        "relative_path": "mps.csv",
        "header_row": 1,
        "description": "Metropolitan Police crime by borough, crime category, and month.",
    },
    {
        "dataset_name": "fly_tipping",
        "table_name": "bronze_fly_tipping",
        "relative_path": "flytipping.csv",
        "header_row": 1,
        "description": "Fly-tipping incidents and enforcement actions by borough and financial year.",
    },
    {
        "dataset_name": "csi",
        "table_name": "bronze_csi",
        "relative_path": "csi.csv",
        "header_row": 6,
        "description": "CSI combined scores; Silver derives borough-level civic-strength context features.",
    },
    {
        "dataset_name": "imd",
        "table_name": "bronze_imd",
        "relative_path": "imd.csv",
        "header_row": 1,
        "description": "IMD 2019 borough domain summaries used as static context features.",
    },
    {
        "dataset_name": "income",
        "table_name": "bronze_income",
        "relative_path": "income.csv",
        "header_row": 0,
        "description": "Annual taxpayer income extract with a two-row header retained for Silver parsing.",
    },
    {
        "dataset_name": "population",
        "table_name": "bronze_population",
        "relative_path": "population.csv",
        "header_row": 1,
        "description": "ONS MYEB3 mid-year population estimates and components by local authority.",
    },
    {
        "dataset_name": "unemployment",
        "table_name": "bronze_unemployment",
        "relative_path": "unemployment.csv",
        "header_row": 1,
        "description": "Unemployment raw-long extract from annual economic activity sheets.",
    },
]


## 3. Helper Functions


In [ ]:
def clean_column_name(name: str, position: int) -> str:
    """Return a Delta column name. Empty names become col_(position)."""
    if name is None or not str(name).strip():
        base = f"col_{position}"
    else:
        base = str(name).strip().lower()
        base = re.sub(r"[^a-z0-9]+", "_", base)
        base = re.sub(r"_+", "_", base).strip("_")
        if not base:
            base = f"col_{position}"
    if base[0].isdigit():
        base = f"c_{base}"
    return base


def make_unique_column_names(columns):
    """Clean and de-duplicate column names while preserving order from left to right"""
    seen = {}
    result = []

    for idx, col in enumerate(columns, start=1):
        base = clean_column_name(col, idx)
        if base not in seen:
            seen[base] = 1
            result.append(base)
        else:
            seen[base] += 1
            result.append(f"{base}_{seen[base]}")

    return result


def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def read_csv_with_header_row(path: str, header_row: int):
    if header_row < 0:
        raise ValueError("header_row must be 0 or greater. Use 0 to keep all rows with generated column names.")

    if header_row == 0:
        return (
            spark.read
            .option("header", False)
            .option("inferSchema", False)
            .option("multiLine", False)
            .option("quote", '"')
            .option("escape", '"')
            .csv(path)
        )

    if header_row == 1:
        return (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", False)
            .option("quote", '"')
            .option("escape", '"')
            .csv(path)
        )

    # For files with header not on row 1:
    # 1. Read as text and collect header row
    # 2. Read file without header and manually rename columns
    from pyspark.sql import Window
    
    # Read entire file as text to get header
    text_df = spark.read.text(path)
    window = Window.orderBy(F.monotonically_increasing_id())
    text_df = text_df.withColumn("row_num", F.row_number().over(window))
    
    #header row
    header_row_data = text_df.filter(F.col("row_num") == header_row).select("value").first()
    if not header_row_data:
        raise ValueError(f"Could not find header at row {header_row}")
    
    # Parse header CSV line
    import csv
    from io import StringIO
    header_line = header_row_data["value"]
    reader = csv.reader(StringIO(header_line))
    column_names = next(reader)
    
    # Read entire file without header
    df = (
        spark.read
        .option("header", False)
        .option("inferSchema", False)
        .option("multiLine", False)
        .option("quote", '"')
        .option("escape", '"')
        .csv(path)
    )
    
    # Add row numbers and filter to keep only data rows
    df = df.withColumn("_row_num", F.row_number().over(Window.orderBy(F.monotonically_increasing_id())))
    df = df.filter(F.col("_row_num") > header_row).drop("_row_num")

    if len(df.columns) != len(column_names):
        # trim column name to match dataframe
        if len(column_names) < len(df.columns):
            column_names.extend([f"col_{i}" for i in range(len(column_names) + 1, len(df.columns) + 1)])
        else:
            column_names = column_names[:len(df.columns)]
    
    df = df.toDF(*column_names)
    
    return df


def write_bronze_table(dataset_config):
    dataset_name = dataset_config["dataset_name"]
    table_name = dataset_config["table_name"]
    relative_path = dataset_config["relative_path"].lstrip("/")
    header_row = int(dataset_config["header_row"])
    source_path = f"{raw_root.rstrip('/')}/{relative_path}"

    if not path_exists(source_path):
        raise FileNotFoundError(
            f"Source file not found for dataset '{dataset_name}': {source_path}. "
            "Check again"
        )

    df = read_csv_with_header_row(source_path, header_row)
    cleaned_columns = make_unique_column_names(df.columns)
    df = df.toDF(*cleaned_columns)

    ingestion_ts = F.current_timestamp()

    df = (
        df
        .withColumn("_dataset_name", F.lit(dataset_name))
        .withColumn("_source_file", F.lit(relative_path.split("/")[-1]))
        .withColumn("_source_path", F.lit(source_path))
        .withColumn("_header_row", F.lit(header_row))
        .withColumn("_ingestion_ts", ingestion_ts)
    )

    full_table_name = f"{target_namespace}.`{table_name}`"

    (
        df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    row_count = df.count()
    column_count = len(df.columns)

    return {
        "dataset_name": dataset_name,
        "table_name": table_name,
        "source_path": source_path,
        "target_table": full_table_name,
        "header_row": header_row,
        "row_count": row_count,
        "column_count": column_count,
        "ingestion_ts": datetime.utcnow(),
    }

## 4. Create Bronze Schema


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_namespace}")

## 5. Ingest Datasets


In [ ]:
audit_rows = []

for dataset in DATASETS:
    print(f"Starting Bronze ingestion: {dataset['dataset_name']}")
    audit = write_bronze_table(dataset)
    audit_rows.append(audit)
    print(f"Completed: {audit['target_table']} | rows={audit['row_count']} | cols={audit['column_count']}")

## 6. Bronze Ingestion Audit Table

In [ ]:
audit_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("header_row", LongType(), False),
    StructField("row_count", LongType(), False),
    StructField("column_count", LongType(), False),
    StructField("ingestion_ts", TimestampType(), False),
])

audit_df = spark.createDataFrame(audit_rows, schema=audit_schema)

# Mask storage account in source_path for display
audit_df_display = audit_df.withColumn(
    "source_path",
    F.regexp_replace("source_path", STORAGE_ACCOUNT, "***")
)

audit_table = f"{target_namespace}.`bronze_ingestion_audit`"
(
    audit_df_display.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(audit_table)
)

display(audit_df_display.orderBy("dataset_name"))

In [ ]:
for dataset in DATASETS:
    full_table_name = f"{target_namespace}.`{dataset['table_name']}`"
    print(f"\n{full_table_name}")
    spark.sql(f"SELECT * FROM {full_table_name} LIMIT 3").display()